# Advanced Tools

Dataset generation, filtering, tokenizer fertility, checkpoint selection, and inference internals. Run these after you have completed the [training notebook](02_train_from_scratch.ipynb).

## Setup

In [ ]:
# ── Environment Setup ────────────────────────────────────────────────────
# Colab: clones the repo and installs dependencies.
# Local / RunPod: assumes you are in the repo root with deps installed.

import sys
from pathlib import Path

if "google.colab" in sys.modules:
    !git clone https://github.com/mkhordoo/dusty-lm.git
    %cd dusty-lm
    !pip install -q uv && uv pip install --system -e .
    !make download-datasets
    print("\u2713 Colab environment ready.")
elif not Path("dustylm").exists():
    raise RuntimeError(
        "Run this notebook from the repository root.\n"
        "  cd /path/to/dusty-lm && jupyter notebook"
    )
else:
    print("\u2713 Running from repo root. Dependencies assumed installed.")

## Inspect the Dataset Files

The training loop expects raw data under `artifacts/datasets/` and tokenized datasets beside it.

In [ ]:
from pathlib import Path

for path in [
    Path("artifacts/datasets/tinystories_base.txt"),
    Path("artifacts/datasets/dusty_pretrain.txt"),
    Path("artifacts/datasets/dusty_sft.jsonl"),
]:
    if path.exists():
        print(f"\u2713 {path}: {path.stat().st_size / 1024 / 1024:.2f} MB")
    else:
        print(f"\u2717 {path}")

## Generate Your Own SFT Dataset

The SFT generator is OpenRouter-backed by default and reads the OpenRouter API key from `OPENAI_API_KEY`. The Makefile currently uses `DUSTY_MODEL=qwen/qwen3-235b-a22b-2507:floor` with `DUSTY_FALLBACK_MODEL=openai/gpt-oss-120b:floor`. In our runs, the Qwen model produced stronger Dusty-style rows, and the `:floor` suffix asks OpenRouter to route to the lowest-cost provider for that model.

You can try different generator models by overriding `DUSTY_MODEL` and `DUSTY_FALLBACK_MODEL`. If you change the assistant's personality, also update the generation prompt, categories, and seed examples in `data_pipeline/generate_sft_dataset_with_fallback.py`. The data generator should describe the exact character you want the trained model to imitate; otherwise the final SFT checkpoint will learn the old Dusty persona even if you rename the model.

Start small. The default full workflow asks for 500 examples per category, which is about 30k candidate rows across the 60 Dusty categories. With the current OpenRouter/Qwen default, generating that scale of data cost us less than $3, but prices and provider routing can change. This demo asks for 5 per category so you can check formatting, rejection rate, and style before spending money on a larger generation run.

In [ ]:
# Optional: set your OpenRouter key before running this cell.
# import os
# os.environ["OPENAI_API_KEY"] = "..."

!make dusty-generate-sft DUSTY_SFT_PER_CATEGORY=5 DUSTY_SFT_BATCH_SIZE=5

## Preview SFT Rows

Each JSONL row needs `category`, `user`, and `dusty`. The `category` field is metadata for filtering and balancing; training uses the user and assistant text.

In [ ]:
import json
from pathlib import Path

sft_path = Path("artifacts/datasets/dusty_sft.jsonl")
with sft_path.open("r", encoding="utf-8") as file:
    for index, line in zip(range(5), file):
        row = json.loads(line)
        print(json.dumps(row, indent=2)[:800])
        print()

## Filter and Balance SFT Data

Filtering removes answers that are too long for a tiny model and samples a category-balanced subset. Use a smaller target for experiments and a larger target for final training.

In [ ]:
!make dusty-filter-sft \
  DUSTY_SFT_FILTER_TARGET=200 \
  DUSTY_SFT_MAX_ANSWER_TOKENS=256 \
  DUSTY_SFT_FILTERED_OUT=artifacts/datasets/dusty_sft_200.jsonl

## Flatten SFT Data into ChatML Pretraining Text

Flattening is useful when you want the tokenizer or pretraining corpus to see the same ChatML boundary tokens used during SFT.

In [ ]:
!make dusty-flatten-sft-pretrain

from pathlib import Path

path = Path("artifacts/datasets/dusty_sft_chatml_pretrain.txt")
print(path)
print(path.read_text(encoding="utf-8")[:1000])

## Tokenizer Fertility

Fertility is tokens per whitespace-delimited word. For a tiny model, roughly `1.2` to `1.5` is usually a useful range. Higher values mean words are split heavily; lower values may spend too much of the model on embeddings.

Vocabulary size is part of the model's parameter budget. A larger vocabulary increases the embedding and output projection matrices, which can make the checkpoint larger without making the transformer layers more capable. Fertility testing asks a practical question: if we spend more parameters on vocabulary, do we actually reduce token count enough to justify it?

For Dusty, moving from about 4k vocabulary to 8k vocabulary did not produce a large enough fertility improvement to justify the extra parameters, so the default Dusty tokenizer stays around 4k tokens.

In [ ]:
!uv run python data_pipeline/tokenizer_fertility_test.py \
  --input artifacts/datasets/tinystories_base.txt \
  --training-files artifacts/datasets/tinystories_base.txt artifacts/datasets/dusty_sft_chatml_pretrain.txt \
  --lines 10000 \
  --vocab-sizes 2048 4096 8192

---

## Checkpoint Selection

### Evaluate Pretraining Checkpoints

Before SFT, evaluate the base `dusty8m` checkpoints directly. Pretraining should produce stable local text patterns and a Dusty-like rhythm.

In [ ]:
!make dusty-generate PROFILE=dusty8m PROMPT="i wake up." TOP_P=0.9 TEMPERATURE=0.8

### Compare Base Step Checkpoints

Use `CHECKPOINT_STEP` to sample intermediate pretraining checkpoints. Pick the best base checkpoint before SFT, because `sft_dusty8m` initializes from `artifacts/checkpoints/dusty8m.pt`.

In [ ]:
# Use story-based cliffhangers to test the raw pre-trained base model
base_prompts = [
    "Once upon a time,",
    "Lily was a little girl who loved to",
    "Deep in the forest, there lived a little boy named Timmy. One day, he",
]
base_checkpoint_steps = [50, 100, 150]

for step in base_checkpoint_steps:
    print("#" * 80)
    print("BASE CHECKPOINT_STEP:", step)
    for prompt in base_prompts:
        print("=" * 80)
        print("PROMPT:", prompt)
        !make dusty-generate PROFILE=dusty8m CHECKPOINT_STEP={step} PROMPT="{prompt}" TOP_P=0.9 TEMPERATURE=0.8

### Promote the Selected Base Checkpoint

Set `BEST_PRETRAIN_STEP` to the best base checkpoint. This overwrites `artifacts/checkpoints/dusty8m.pt`, so the later SFT run starts from the selected base model.

In [ ]:
BEST_PRETRAIN_STEP = 100

!cp artifacts/checkpoints/dusty8m_step_{BEST_PRETRAIN_STEP}.pt artifacts/checkpoints/dusty8m.pt

# This command intentionally omits CHECKPOINT_STEP. It now uses the promoted base checkpoint.
!make dusty-generate PROFILE=dusty8m PROMPT="i wake up." TOP_P=0.9 TEMPERATURE=0.8

### Choose an SFT Checkpoint by Generation

Loss is only a coarse signal for tiny character models. During training, step checkpoints such as `artifacts/checkpoints/dusty8m_sft_step_100.pt` let you evaluate intermediate model behavior. Generate the same fixed prompt set from several `CHECKPOINT_STEP` values and compare instruction following, persona consistency, repetition, and answer length.

Unlike the pre-training phase, Supervised Fine-Tuning (SFT) does not follow Chinchilla scaling laws. Instead of aiming for hundreds of millions of tokens, SFT is simply about teaching the base model a specific conversational format and persona. For this repository, we use a custom 30k instruction dataset. We found that training for about 7 epochs (roughly 1,960 iterations, depending on your batch size) is the sweet spot.

During SFT, the model can overfit to the chat format quickly. This is why you should stop relying purely on the loss curve and begin selecting checkpoints qualitatively by reading the generated text.

When a step checkpoint behaves better than the final checkpoint, promote it by copying that file over the profile's default checkpoint path. After promotion, remove `CHECKPOINT_STEP` from the generation command; normal `make dusty-generate PROFILE=sft_dusty8m ...` will use the selected model.

In [ ]:
prompts = [
    "where are you?",
    "are you scared?",
    "what do you dream about?",
    "where do you charge?",
]

checkpoint_steps = [50, 100, 150]

for step in checkpoint_steps:
    print("#" * 80)
    print("CHECKPOINT_STEP:", step)
    for prompt in prompts:
        print("=" * 80)
        print("PROMPT:", prompt)
        !make dusty-generate PROFILE=sft_dusty8m CHECKPOINT_STEP={step} PROMPT="{prompt}" TOP_P=0.9 TEMPERATURE=0.8

### Promote the Selected SFT Checkpoint

Set `BEST_STEP` to the checkpoint step that produced the best qualitative behavior. This overwrites `artifacts/checkpoints/dusty8m_sft.pt`, which is the default checkpoint for the `sft_dusty8m` profile.

In [ ]:
BEST_STEP = 100

!cp artifacts/checkpoints/dusty8m_sft_step_{BEST_STEP}.pt artifacts/checkpoints/dusty8m_sft.pt

# This command intentionally omits CHECKPOINT_STEP. It now uses the promoted default checkpoint.
!make dusty-generate PROFILE=sft_dusty8m PROMPT="where are you?" TOP_P=0.9 TEMPERATURE=0.8

---

## Inference Internals

The quickstart notebook hides the response object and runtime profile logic. This section shows the lower-level inference API, context-window controls, and checkpoint profile detection used by DustyLM.

### Load an Inference Engine

Use the local promoted SFT checkpoint if you trained one. If you are only inspecting a downloaded checkpoint, set `checkpoint_path` and `tokenizer_path` to those files instead.

In [ ]:
from pathlib import Path
import torch
from dustylm.inference import Inference

checkpoint_path = Path("artifacts/checkpoints/dusty8m_sft.pt")
tokenizer_path = Path("artifacts/tokenizers/dusty_tokenizer.json")

engine = Inference(
    checkpoint_path=checkpoint_path,
    tokenizer_path=tokenizer_path,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print("Profile:", engine.profile_name)

### OpenAI-Style Chat Completion

`Inference.chat_completion()` accepts OpenAI-style `messages` and returns an OpenAI-like dictionary. That keeps local DustyLM inference close to common chat-completion calling patterns while still running against a local PyTorch checkpoint.

In [ ]:
messages = [{"role": "user", "content": "where are you?"}]

response = engine.chat_completion(
    messages=messages,
    max_tokens=64,
    temperature=0.8,
    top_p=0.8,
)

response

In [ ]:
assistant_text = response["choices"][0]["message"]["content"]
print("Dusty>", assistant_text)

### Short Context Windows for Tiny Models

Dusty is only about 8M parameters, so the default chat history window is intentionally tiny: `max_chat_turns=1`. That keeps the current user request from being diluted by old conversation tokens. Larger SFT profiles, such as SmolLM2-based models, can use a larger window.

In [ ]:
print("Profile:", engine.profile_name)
print("Default max_chat_turns:", engine.spec.max_chat_turns)

history = [
    {"role": "system", "content": "Answer as Dusty, a tiny vacuum robot."},
    {"role": "user", "content": "what is under the couch?"},
    {"role": "assistant", "content": "crumbs. many crumbs."},
    {"role": "user", "content": "should you go there?"},
]

response = engine.chat_completion(history, max_tokens=64)
print("Default window:", response["choices"][0]["message"]["content"])

response = engine.chat_completion(history, max_tokens=64, max_chat_turns=2)
print("Two-turn window:", response["choices"][0]["message"]["content"])

### Convenience Wrapper and CLI Chat

For scripts or demos, wrap the response dictionary and return only the generated assistant text. For a terminal chat loop, use `make chat`, which runs `uv run python -m dustylm.inference`.

In [ ]:
def chat(prompt, *, max_tokens=64, temperature=0.8, top_p=0.8):
    response = engine.chat_completion(
        [{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    return response["choices"][0]["message"]["content"].strip()


for prompt in [
    "who are you?",
    "are you scared?",
    "what do you dream about?",
    "go charge",
]:
    print(f"You> {prompt}")
    print(f"Dusty> {chat(prompt)}\n")